In [ ]:
"""
Runs DDD model either in single run model or for calibration
The model itself is called as a function which calls on several functions
This one is for ECCO use not MRT and not operative or HSO
"""
#using CSV
using Distributions
using LsqFit
using Statistics
using Dates
using DataFrames
using Plots
using CSV
using BlackBoxOptim
using JLD2

##Preprocessing routines
include("..\\DDDFunctions\\Big2SmallLambda.jl")
#include("..\\DDDFunctions\\CeleritySubSurface_DupBous.jl")
include("..\\DDDFunctions\\CeleritySubSurface.jl")
include("..\\DDDFunctions\\SingleUH.jl")
include("..\\DDDFunctions\\SingleNormalUH.jl")
include("..\\DDDFunctions\\LayerEstimation.jl")
include("..\\DDDFunctions\\PyrAreas.jl")
include("..\\DDDFunctions\\GrWPoint.jl")
include("..\\DDDFunctions\\RiverPoint.jl")
include("..\\DDDFunctions\\TemperatureVector.jl")

##EB and Snow Routines
include("..\\DDDFunctions\\NedbEBGlac_debug04072022.jl")
include("..\\DDDFunctions\\SnowpackTemp.jl")
include("..\\DDDFunctions\\TempstartUpdate.jl")
include("..\\DDDFunctions\\SmeltEBGlac_debug04072022.jl")
include("..\\DDDFunctions\\CloudCoverGlac_debug04072022.jl")
include("..\\DDDFunctions\\TssDewpoint.jl")
include("..\\DDDFunctions\\SolradTransAlbedoper_hrs_debug04072022.jl")
include("..\\DDDFunctions\\LongWaveRad_debug04072022.jl")
include("..\\DDDFunctions\\SensibleLatHeat_debug04072022.jl")
include("..\\DDDFunctions\\AlbedoUEB_debug04072022.jl")
include("..\\DDDFunctions\\GroundPrecCC.jl")
include("..\\DDDFunctions\\SnowGamma.jl")
include("..\\DDDFunctions\\Varc.jl")
include("..\\DDDFunctions\\NewSnowDensityEB.jl")
include("..\\DDDFunctions\\NewSnowSDEB.jl")
include("..\\DDDFunctions\\DensityAge.jl")

#Subsurface and Evaporation routines
include("..\\DDDFunctions\\LayerCapacityUpdate.jl")
include("..\\DDDFunctions\\PotentialEvapPT.jl")
include("..\\DDDFunctions\\UnsaturatedEvapEB.jl")
include("..\\DDDFunctions\\LayerEvap.jl")
include("..\\DDDFunctions\\UnsaturatedExEvap.jl")
include("..\\DDDFunctions\\WetlandsEB.jl")
include("..\\DDDFunctions\\GrvInputDistributionICap2022.jl")
include("..\\DDDFunctions\\OFICap.jl")
include("..\\DDDFunctions\\LayerUpdate.jl")
include("..\\DDDFunctions\\BogLayerUpdate.jl")
include("..\\DDDFunctions\\RiverUpdate.jl")
## Overland Flow routine
include("..\\DDDFunctions\\OverlandFlowDynamicDD.jl")
## Efficiency criteria
include("..\\DDDFunctions\\NSE_ths.jl")
include("..\\DDDFunctions\\KGE_ths.jl")
# Model Module
#include("F:\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\DDDUrbanFunc.jl")
include("..\\DDDFunctions\\DDDAllTerrain22012024.jl") #This one for ECCO use not MRT #This one for ECCO use not MRT
########################################################################################

# navn = "Austdalsvatnet"
# catchment = "Austdalsvatnet"

# navn = "Tunsbergdalsvatnet"
# catchment = "Tunsbergdalsvatnet"

# navn = "Gaupne"
# catchment = "Gaupne"

# navn = "Gaupne_example"
# catchment = "Gaupne_example"

# navn = "Myklemyr_test"
# catchment = "Myklemyr_test"

# navn = "Geisdalsvatnet"
# catchment = "Geisdalsvatnet"

# navn = "BåtedølaS"
# catchment = "BåtedølaS"

# navn = "BåtedølaS_test"
# catchment = "BåtedølaS_test"

navn = "Nigardsbrevatnet"
catchment = "Nigardsbrevatnet"

# navn = "Fonndøla"
# catchment = "Fonndøla"

# navn = "Nigard"
# catchment = "Nigard"




TR = "1H"         # this is just a marker for naming files, does NOT set the temporal resolution

ptqfile = string("parameters&ptqs\\", navn, "_", TR,"_ptq_edited.csv") 
# ptqfile = string("parameters&ptqs\\", navn, "_", TR,"_ptq_edited_20.csv") 
# ptqfile = string("parameters&ptqs\\finished_ENS_ptqs\\Austdalsvatnet_EC_ENS_20181008T00_member_10_merged_ptq.csv") 

#ptqfile = string("parameters&ptqs\\split_ptqs\\BåtedølaS_2018") # just BåtedølaS 2018

r2fil = string("outputs_Jostedal\\", navn,"_r2_",TR,".csv")

# utfile = string("outputs_Jostedal\\", navn,"_utfile_",TR,".csv") 
utfile = string("outputs_Jostedal\\", navn,"_utfile_",TR,".csv") 

paramfile = string("parameters&ptqs\\", navn, "_", TR,"_param_edited.csv")

spinup = (31*4) #days used to spin up the model. 

prm = CSV.read(paramfile,DataFrame,header=["Name", "val"], delim=';')
#prm = CSV.read(paramfile,header=["Name", "val"], delim=';')

#            u,          pro           TX,         Pkorr        skorr,     GscInt      OVP          
#         OVIP       Lv            rv        
tprm = [prm.val[20], prm.val[21], prm.val[22], prm.val[18], prm.val[19],prm.val[33], prm.val[34], 
    prm.val[35],prm.val[36],prm.val[37]]

# # println(tprm)

# tprm = [1.86393, 0.05, -0.478436, 1.19771, 1.90961, 0.008, 0.01, 0.01, 0.01, 1.57179] # Bruvollelvi + Fonndøla 
# tprm = [1.86393, 0.05, -0.7, 1.2, 1.90961, 0.006, 0.01, 0.01, 0.01, 1.57179] # Geisdalsvatnet

# tprm = [2, 0.0125258319845122, 0.5, 0.9, 1.2833769181048194, 0.007, 0.02, 0.06, 0.1, 1.2] # Myklemyr (best på flomtopp)
# tprm = [1.86393, 0.05, -0.478436, 1.19771, 1.90961, 0.008, 0.01, 0.01, 0.06, 1.57179] # Nigardsbrevatnet
# tprm = [2.17, 0.1, 0.24, 1.19771, 1.190961, 0.02, 0.02, 0.03, 0.0134162, 1.57179] # Nigardsbrevatnet testing new things

# tprm=[2.086461098353537, 0.0119921973927499, 1.5554023242009132, 1.52916028, 1.4430364901632804,
#     0.0036997114120818713, 0.007, 0.01, 0.01, 1.0] # Nigardsbrevatn HSO parameters

# tprm = [2.133087102893902, 0.0126691512871639, 0.8909068021308982, 1.52513932, 
#     1.5597672009194696, 0.0050497954350014, 0.007, 0.01, 0.01, 1.0] # Tunsbergdalsvatn HSO parameters



# Gshape, Gscale = Big2SmallLambda(prm.val[32], prm.val[33]) # Coverting integrated celerity to layers takes too long in calibration: preprocessing
Gshape, Gscale = Big2SmallLambda(prm.val[32], tprm[6])
Gpar = [Gshape, Gscale]

println(prm.val[32]," ", prm.val[33])
println("Gshape= ", prm.val[32]," Gscale= ", tprm[6])

startsim = 1 
kal = 0
modstate = 0
savestate = 0

t1= time_ns()

function calib_wrapper_model(Gpar,startsim, tprm, prm, ptqfile, utfile, r2fil, modstate, savestate, kal, spinup)
 qobs, qberegn, KGE, NSE, bias = DDDAllTerrain(Gpar,startsim, tprm, prm, ptqfile, utfile, r2fil, modstate, savestate,
        kal, spinup)  
 return qobs,qberegn, KGE,NSE,bias 
end


function calib_single_wsh(Gpar,startsim, tprm, prm, ptqfile, utfile, r2fil, modstate, savestate, kal, spinup)
 qobs, qberegn, KGE, NSE, bias = DDDAllTerrain(Gpar,startsim, tprm, prm, ptqfile, utfile, r2fil, modstate, savestate,
        kal, spinup)    
 return (1.0 - KGE)
end

# OLD
# if(kal == 0)
#     qberegn = calib_wrapper_model(Gpar,startsim, tprm, prm, ptqfile, utfile, r2fil,
#         modstate, savestate,kal, spinup) # a single run 
    
#     println(catchment)
#     println("KGE=",round(KGE,digits=3))
#     println("NSE=",round(NSE,digits=3))
#     println("bias=",round(bias,digits=3))
# end

# NEW
if kal == 0
    qobs, qberegn, KGE, NSE, bias = calib_wrapper_model(
        Gpar, startsim, tprm, prm, ptqfile, utfile, r2fil,
        modstate, savestate, kal, spinup)

    println(catchment)
    println("KGE=", round(KGE,digits=3))
    println("NSE=", round(NSE,digits=3))
    println("bias=", round(bias,digits=3))
end



if(kal == 1) # calibrate
#     #                   u,        pro,         TX,        Pkorr,    skorr,          GscInt,         OVP      
#     param_range = [(1.0,3.0), (0.05,0.05), (-0.5, 0.5), (0.5, 2.0), (0.5,2.0), (0.065,0.075), (tprm[7],tprm[7]),
#     #        OVIP               Lv                 rv
#         (tprm[8],tprm[8]), (tprm[9],tprm[9]),(tprm[10],tprm[10])] 
    
    param_range = [(1.86393,1.86393), (0.0187,0.0187), (0.24, 0.24), (1.19, 2.0), (1.12, 1.12), (0.008,0.008), (0.007,0.007),
        (tprm[8],tprm[8]), (0.01,0.07),(1.3,1.3)] # 
    
    println(param_range)
    calib_single_wsh_tmp(param) = calib_single_wsh(Gpar,startsim, param, prm, ptqfile, utfile, r2fil,
                                           modstate, savestate, kal, spinup)
    res = bboptimize(calib_single_wsh_tmp; SearchRange = param_range, MaxSteps = 100, TraceMode = :verbose)
    param_hydro = best_candidate(res)
    println(param_hydro)
end

t2 = time_ns()
println("Pkorr=", round(tprm[4],digits=3))
println("Time elapsed[s]= ",(t2-t1)/1.0e9)
println("tprm:", tprm)

if(kal==0)
#  plot(qobs[6500:8000], color="black",label = "Observed",lw =1)
#  plot!(qberegn[6500:8000],col= "blue",label = "Simulated", lw = 2)
 plot(qobs[40000:46000], color="black",label = "Observed",lw =1)
 println(maximum(qberegn[40000:46000]))
 plot!(qberegn[40000:46000],col= "blue",label = "Simulated", alpha=0.6, lw = 2)
end 
 

1.0 0.0005467692501852
Gshape= 1.0 Gscale= 0.0005467692501852


In [2]:
# 1. Find indices where observed flow exceeds threshold
idx = qobs .> 1.2#1.53 # 5-year flood level (Bruvollelvi*0.282)

# 2. Extract corresponding values
qobs_flood = qobs[idx]
qsim_flood = qberegn[idx]
println(qobs_flood)
println(qsim_flood)
# 3. Compute RMSE
rmse = sqrt(mean((qsim_flood .- qobs_flood).^2))

println("RMSE for flows > 1.53 m³/s = ", rmse)


 plot(qobs[1000:13710], color="black",label = "Observed")
 plot!(qberegn[1000:13710],col= "blue",alpha=0.5, label = "Simulated")
#[40726:43204]


LoadError: UndefVarError: `qobs` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [8]:
#function CeleritySubSurface(NoL, Gshape, Gscale, midDL, Timeresinsec) 
 using Distributions  
 using Plots
  NoL = 5
  Gshape = 4.0
  meanVel =0.00012
  midDL = 129.84 
  Timeresinsec = 10800
  a = 3 # rapid layer multiplier for DF aquifers
  Gscale = meanVel*Timeresinsec/(Gshape*midDL) 

k = zeros(Float64,NoL)                   #celerity of subsurface (and overland) flow
probvec = zeros(Float64,NoL)            #all leves and overland flow level

dp = 1/(NoL-1)                              #Overland flow level (nol=1], fixed celerity and extremely high capacity(2000 mm)

for i in reverse(1:(NoL-1))
    probvec[i+1] = i*dp - dp/2                #celerities are estimated at center of level, hence dp/2
end

probvec = 1 .- probvec
probvec[1] = 0.99                            #Quantile in celerity distribution for overland flow fixed at 0.99

g = Gamma(Gshape, Gscale)    
k[1:NoL] = quantile.(g,probvec[1:NoL])*midDL/Timeresinsec
##Dupuit Forschheimer 
k[NoL] = 4*meanVel/(3+a) 
k[4] = 4*meanVel/(3+a) 
k[3] = 4*meanVel/(3+a) 
k[2] = k[3]*a
k[1] = 0.0005

#plot(k[1:NoL], color="black",label = string("Shape=",Gshape),lw =1)
#plot!(k[1:NoL], color="green",label = string("Shape=",Gshape),lw =1)
#plot!(k[1:NoL], color="red",label = string("DF"),lw =3)
plot!(qberegn[5000:7000],col= "blue",label = "Simulated", lw = 2)

#return k                        #Celerities [m/s] 
#end

LoadError: BoundsError: attempt to access Tuple{Vector{Float64}, Vector{Float64}, Float64, Float64, Float64} at index [5000]

In [7]:
 using Distributions
 include("..\\DDDFunctions\\SingleUH.jl")
#Røykenes
  GshInt = 0.8
  GscInt = 0.055 
  midDL = 156.07
  maxDl = 928
  MAD = 5.24
  Timeresinsec = 10800
  NoL = 2
  area2 = 50090000 
  gtcel = 0.99
 
mLam = GshInt*GscInt
varLam = GshInt*(GscInt)^2                           #Yevjevich p.145
meanIntk = mLam*midDL/Timeresinsec                   #mean celerity estimated through Integrated Celerity
antBox = Int(trunc(maxDl/(meanIntk*Timeresinsec)))+1 #Temporal length UH_MAD
UH_MAD = zeros(Float64,antBox)
sRes = zeros(Float64,antBox) # saturation sum

#Unit hydrograph for MAD
UH_MAD = SingleUH(meanIntk,Timeresinsec, midDL, maxDl, 0)

StSt = (1000*MAD*Timeresinsec)/(area2)         # Steady state Input eq. output in mm
sRes[1] = 0
sRes[2:antBox] .= StSt.*UH_MAD[2:antBox]

for i in 3: antBox
  sRes[i:antBox] .= sRes[i:antBox] + StSt.*UH_MAD[i:antBox]
end

mRes = sum(sRes)
Fact = mLam/mRes
stdRes = (varLam/Fact^2)^0.5                   # see Haan p.51

GshRes =  1.83 #mRes^2/stdRes^2
GscRes =  11.43 #stdRes^2/mRes

MLev = [1/(NoL-1):1/(NoL-1):1.0;]              # (sequence)Quantiles  to calculate reservoir levels [0.1:0.1:0.9;]

MLev[NoL-1] = gtcel                            # quantile for start overland flow
Res_prob = zeros(Float64,(NoL-1))
Magkap = zeros(Float64,NoL)
g = Gamma(GshRes,GscRes) 
#calculates the reservoir levels associated with quantiles. Mean is GshRes*GscRes
Res_prob .= quantile.(g,MLev)

#Capasity of Layers
ssRes1 = zeros(Float64,NoL)
ssRes1[1] = 2000                               # capacity of overland flow level

for i in 2:(NoL-1)
  ssRes1[i] = Res_prob[NoL-i+1]-Res_prob[(NoL-i)]
end

ssRes1[NoL] = Res_prob[1]                     # capasity for the first slowest level         
                       
Magkap = ssRes1                                 # capasity for Layers
M = Res_prob[(NoL-1)]                           # Total groundwater reservoir
println(GshRes) 
println(GscRes)
println("Magkap fra Subrutine ",Magkap)
println("M fra Subrutine ", M)

1.83
11.43
Magkap fra Subrutine [2000.0, 72.21658698613973]
M fra Subrutine 72.21658698613973


In [8]:
NoL = 2
GshRes =  1.83 #mRes^2/stdRes^2
GscRes =  11.43 #stdRes^2/mRes
gtcel = 0.99

MLev = [1/(NoL-1):1/(NoL-1):1.0;]              # (sequence)Quantiles  to calculate reservoir levels [0.1:0.1:0.9;]

MLev[NoL-1] = gtcel                            # quantile for start overland flow
Res_prob = zeros(Float64,(NoL-1))
Magkap = zeros(Float64,NoL)
g = Gamma(GshRes,GscRes) 
#calculates the reservoir levels associated with quantiles. Mean is GshRes*GscRes
Res_prob .= quantile.(g,MLev)

#Capasity of Layers
ssRes1 = zeros(Float64,NoL)
ssRes1[1] = 2000                               # capacity of overland flow level

for i in 2:(NoL-1)
  ssRes1[i] = Res_prob[NoL-i+1]-Res_prob[(NoL-i)]
end

ssRes1[NoL] = Res_prob[1]                     # capasity for the first slowest level         
                       
Magkap = ssRes1                                 # capasity for Layers
M = Res_prob[(NoL-1)]                           # Total groundwater reservoir
println(GshRes) 
println(GscRes)
println("Magkap fra Subrutine ",Magkap)
println("M fra Subrutine ", M)

1.83
11.43
Magkap fra Subrutine [2000.0, 72.21658698613973]
M fra Subrutine 72.21658698613973
